In [1]:
import pathlib
from pathlib import Path
import pandas as pd
file_path = pathlib.Path("raw_data","wwtp_data.csv")
ad_facility = pd.read_csv(file_path)


In [4]:
df = ad_facility[["state","city","has_ad","flow_m3_per_day"]]

In [5]:
df["plant_flow_ad"] = df["flow_m3_per_day"].where(df["has_ad"]==str("yes")) 

In [6]:
df = df.dropna(axis=0,how="any")

In [7]:
#physical constants
MMBTU_TO_MJ = 1055.06 # 1mmBtu = 1055.06MJ a conversion 
HHV1 = 55.6 # MJ/KgCH4 (El Abbadi, Preprint)
eff = 0.90 # Methane recovery effficiency (%) from Parker etal.
mc = 0.65 # Methane content in biogas (%) El Abbadi preprint
p = 1.15 # Density of biogas in kg/m3(Jørgensen,2009)​
p1 = 0.656 # Density of methane in kg/m3(Speight,2020)​


In [8]:
#tecno-economic assumptions
cf = 8322 # 95% Capacity Factor(cf) in hours
t = 20 # lifetime technology in years 
wacc = 0.1 # WACC is the Weighted Average Cost of Capital (WACC). A 10% WACC is assumed. Will refine later

In [9]:
def calc_biogas_flow(flow_rate): 
    """Calculates biogas flowrate in KgCH4/h from El Abbadi preprint
    This function takes in the whole dataframe as input but selects 
    the plant_flow _ad column which is the facilities flowrate in m3/day
    """
    #df["biogas_flow_kg_h"] = df["plant_flow_ad"]*0.00148
    return flow_rate*0.00148

df["biogas_flow_kg_h"] = df["plant_flow_ad"].apply(calc_biogas_flow)


In [10]:
def calc_biogas_flow_m3(biogas_flow_kg_h,p):
    """Calculates biogas flowrate in m3/h using density of biogas(p)
    and biogas_flow_kg_h column from the df as inputs    
    """
    #df["capacity_m3_h"] = df["biogas_flow_kg_h"]/p
    return biogas_flow_kg_h/p

df["capacity_m3_h"] = df["biogas_flow_kg_h"].apply(calc_biogas_flow_m3, args=(p,))

In [11]:
def calc_rng_flow_m3(capacity_m3_h,mc,eff):
    """Calculates RNG flow in m3/h using methane content(mc) in %,
    methane recovery efficiency(eff) in % and capacity_m3_h column
    from the df as inputs    
    """
    #df["rng_m3_h"] = df["capacity_m3_h"]*mc*eff
    return capacity_m3_h*mc*eff

df["rng_m3_h"] = df["capacity_m3_h"].apply(calc_rng_flow_m3, args=(mc,eff,)) 

nnn = calc_rng_flow_m3(2.435830,0.65,0.9) # Reason to use .apply. Thank you Prof El Abbadi :)

In [12]:
def calc_rng_flow_mmbtu(rng_m3_h,p1,HHV1,MMBTU_TO_MJ):
    """Calculates RNG flow in mmbtu/h using density of methane(p1) in kg/m3,
    Higher Heating Value(HHV1) of methane in MJ/KgCH4, the conversion factor for 
    mmBtu to MJ(MMBTU_TO_MJ) and rng_m3_h column from the df as inputs    
    """
    #df["rng_mmbtu_h"] = (df["rng_m3_h"]*p1*HHV1)/MMBTU_TO_MJ
    return (rng_m3_h*p1*HHV1)/MMBTU_TO_MJ

df["rng_mmbtu_h"] = df["rng_m3_h"].apply(calc_rng_flow_mmbtu, args=(p1,HHV1,MMBTU_TO_MJ,))

In [13]:
def calc_rng_flow_mmbtu_y(rng_mmbtu_h,cf):
    """Calculates RNG flow in mmbtu/yr using Capacity Factor(cf)
    and rng_mmbtu_h column from the df as inputs    
    """
    return rng_mmbtu_h*cf

df["rng_mmbtu_yr"] = df["rng_mmbtu_h"].apply(calc_rng_flow_mmbtu_y, args=(cf,))

In [14]:
def calc_capex(rng_mmbtu_h):
    """Calculates Capex($) using Parker etal model as well as
    rng_mmbtu_h column from the df as inputs
    """
    #df["Capex($)"] = ((df["rng_mmbtu_h"])**0.56)*1370000
    return (rng_mmbtu_h**0.56)*1370000

df["Capex($)"] = df["rng_mmbtu_h"].apply(calc_capex)

In [15]:
def calc_om_cost(rng_mmbtu_h):
    """Calculates O&M cost($/yr) using Parker etal model as well as
    rng_mmbtu_h column from the df as inputs
    """
    return (rng_mmbtu_h**0.81)*101625

df["O&M($/yr)"] = df["rng_mmbtu_h"].apply(calc_om_cost)


In [16]:
def calc_vc_cost(rng_mmbtu_h):
    """Calculates variable cost($/yr) using
    rng_mmbtu_h column from the df as inputs
    """
    # This is assuming variable cost is zero for now
    return (rng_mmbtu_h**0)*0

df["Variable Cost($/yr)"] = df["rng_mmbtu_h"].apply(calc_vc_cost)

In [17]:
"""The next cells of codes calculates levelized Capex,O&M cost
and variable cost which sums up as the total levelized cost
"""

'The next cells of codes calculates levelized Capex,O&M cost\nand variable cost which sums up as the total levelized cost\n'

In [18]:
def calc_crf(t,wacc):
    """Calculates CRF using equation from (El Abbadi etal 2022)"""
    crf = wacc/(1-(1+wacc)**-t)
    return crf
crf = calc_crf(t,wacc)
print("CRF = ",crf)

CRF =  0.11745962477254576


In [19]:
def calc_unit_capex(capex,rng_mmbtu_yr):
    """Calculates unit capital cost in $/(mmBtu/yr) using equation 
    from (El Abbadi etal 2022) as well as Capex and 
    rng_mmbtu_yr column from the df as inputs
    """
    return capex/rng_mmbtu_yr

df["Unit Capex($/(mmBtu/yr))"] = df[""]
    
annual_capex = cal_annual_capex(capex,crf)
print("Annualized Capex($/yr) = ",annual_capex)

KeyError: ''

In [20]:
def calc_lcapex_cost(rng_mmbtu_yr):
    """Calculates O&M cost($/yr) using Parker etal model as well as
    rng_mmbtu_h column from the df as inputs
    """
    return (rng_mmbtu_h**0.81)*101625

df["O&M($/yr)"] = df["rng_mmbtu_h"].apply(calc_om_cost)